In [ ]:
# Install the Hugging Face Transformers library
!pip install transformers

In [ ]:
from google.colab import userdata

# Retrieve the Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

if hf_token:
    print("Hugging Face token retrieved successfully.")
else:
    print("Error: Hugging Face token not found. Please add 'HF_TOKEN' to Colab secrets (under the key icon in the left sidebar).")

Hugging Face token retrieved successfully.


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from google.colab import userdata # Import userdata here

# Retrieve the Hugging Face token from Colab secrets to ensure it's defined
hf_token = userdata.get('HF_TOKEN')

# Load a pre-trained instruction-tuned model and tokenizer for better general knowledge QA.
# 'google/gemma-2b-it' is a good choice for factual question answering and instruction following.
# Using device_map='auto' to utilize GPU if available.
print("Loading google/gemma-2b-it model and tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it", token=hf_token)
    model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto", token=hf_token)
    print("Model and tokenizer loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("This is likely due to the model being gated. Please ensure you have accepted the terms and conditions for 'google/gemma-2b-it' on Hugging Face (https://huggingface.co/google/gemma-2b-it) and that your HF_TOKEN has read access.")
    model = None # Set model to None if loading fails to indicate its absence
    tokenizer = None # Set tokenizer to None if loading fails

Loading google/gemma-2b-it model and tokenizer...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model and tokenizer loaded successfully.


In [6]:
import torch
import textwrap # Import textwrap for formatting output

# Keep track of conversation history as a list of dictionaries for the chat template
chat_history = []

print("\n--- Chatbot Started ---")
print("Type 'exit' or 'quit' to end the conversation.")

# Set a width for wrapping chatbot responses
RESPONSE_WRAP_WIDTH = 80 # You can adjust this value if needed

# Check if the model and tokenizer were loaded successfully
if model is None or tokenizer is None:
    print("\nERROR: Model and/or tokenizer could not be loaded. Please check the previous cell's output and ensure you have accepted the Hugging Face terms for 'google/gemma-2b-it'. Cannot provide factual answers without the correct model.")
else:
    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break

        # Add user message to chat history
        chat_history.append({"role": "user", "content": user_input})

        # Apply chat template to prepare input for the model, including generation prompt
        processed_input = tokenizer.apply_chat_template(
            chat_history,
            tokenize=True,
            return_tensors="pt",
            add_generation_prompt=True # Crucial for instruction-tuned models to know it's their turn
        ).to(model.device)

        input_ids = processed_input['input_ids']
        attention_mask = processed_input['attention_mask']

        # Generate a response from the model
        output = model.generate(
            input_ids,
            attention_mask=attention_mask, # Pass attention_mask explicitly
            max_new_tokens=250,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode only the newly generated part of the response
        response_tokens = output[0, input_ids.shape[-1]:]
        response = tokenizer.decode(response_tokens, skip_special_tokens=True)

        # Wrap the chatbot's response for better readability
        wrapped_response = textwrap.fill(response, width=RESPONSE_WRAP_WIDTH)
        print(f"Chatbot: {wrapped_response}")

        # Add bot's response to chat history for context in the next turn
        chat_history.append({"role": "model", "content": response})

print("--- Chatbot Ended ---")


--- Chatbot Started ---
Type 'exit' or 'quit' to end the conversation.

You: Hello, Good Morning
Chatbot: Good Morning! I hope you have a wonderful day ahead. How can I help you today?

You: what is java?
Chatbot: Java is a widely used programming language known for its versatility and ease of
use. It is often used for developing applications for various platforms,
including mobile, web, and desktop software. Java is a robust language with a
rich set of features and libraries that make it a popular choice for developing
complex and scalable software solutions.

You: then python?
Chatbot: Sure, here is a brief overview of Python:  * **High-level:** Python is a high-
level programming language, making it easier to learn than some other languages.
* **Dynamic typing:** Python is dynamically typed, meaning you don't need to
explicitly declare the type of variables before using them. * **Interpreted:**
Python is an interpreted language, meaning the code is executed line by line
during runt

# **Project Summary & Insights**

### **1. Model Choice & Implementation**
For this task, I implemented the `google/gemma-2b-it` model. While models like *DialoGPT* are optimized for casual "chit-chat," **Gemma** is an **Instruction-Tuned** model. This ensures that the chatbot provides factual, meaningful responses to technical and general knowledge questions rather than just conversational fillers.

### **2. Conversational Memory (Context Retention)**
A key feature of this implementation is the use of `chat_history`. By passing the previous exchanges back to the model using `tokenizer.apply_chat_template`, the chatbot maintains **Conversation Flow**. This allows the user to ask context-dependent follow-up questions (e.g., asking "then Python?" after a question about Java) without re-specifying the topic.

### **3. Addressing the Knowledge Cutoff**
During testing, I observed that the model could not identify the current Prime Minister of Nepal for the year 2026.
* **Reason:** All Large Language Models (LLMs) have a **"Knowledge Cutoff"**—a specific date beyond which they were not trained.
* **Behavior:** The model correctly exhibited **AI Safety** by refusing to hallucinate or invent a name. Instead, it advised the user to check a reliable news source, which is the expected behavior for a high-quality AI assistant.
* **Future Enhancement:** In a production environment, this limitation would be addressed by integrating **RAG (Retrieval-Augmented Generation)** or a search API to provide real-time data access.

### **4. Technical Conclusion**
The final pipeline successfully handles the **Expected Pipeline Flow**:  
**User Input** $\rightarrow$ **Model Processing** $\rightarrow$ **Response Generation** $\rightarrow$ **Display Output**.  
The system operates in a continuous loop until the mandatory **exit** or **quit** command is received, fulfilling all task requirements.